In [2]:
from binance.client import Client
import pandas as pd
import numpy as np
from datetime import datetime
import time
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ==================== SETTINGS ====================
API_KEY = "https://api1.binance.com/"           
API_SECRET = ""         

# Cryptocurrency settings
SYMBOL = "BTCUSDT"      
INTERVAL = Client.KLINE_INTERVAL_1HOUR  
START_DATE = "2020-01-01"  # Start date for data collection

# File paths
RAW_DATA_FILE = "crypto_raw_data.csv"
PROCESSED_DATA_FILE = "crypto_processed_data.csv"
# ==================================================

# Initializing Binance client
client = Client(API_KEY, API_SECRET)

def download_historical_data(symbol, interval, start_date):
    """
    Download historical cryptocurrency data from Binance
    """
    print(f"\n{'='*50}")
    print(f"Downloading {symbol} data from {start_date}...")
    print(f"{'='*50}")
    
    all_klines = []
    start_ts = int(pd.to_datetime(start_date).timestamp() * 1000)
    
    batch_count = 0
    while True:
        try:
            # To th Get next batch (max 1000 records per request)
            klines = client.get_klines(
                symbol=symbol,
                interval=interval,
                startTime=start_ts,
                limit=1000
            )
            
            if not klines:
                break
            
            all_klines.extend(klines)
            batch_count += 1
            
            # Update start timestamp for next batch
            last_close_time = klines[-1][6]
            start_ts = last_close_time + 1
            
            # Progress indicator
            if batch_count % 10 == 0:
                print(f"Downloaded {len(all_klines)} records...")
            
            # Sleep to avoid rate limits
            time.sleep(0.2)
            
        except Exception as e:
            print(f"Error downloading data: {e}")
            break
    
    print(f"\n✓ Total records downloaded: {len(all_klines)}")
    return all_klines

def create_dataframe(klines):
    """
    Convert raw klines data to pandas DataFrame
    """
    print("\nCreating DataFrame...")
    
    df = pd.DataFrame(klines, columns=[
        "open_time", "open", "high", "low", "close", "volume",
        "close_time", "quote_asset_volume", "number_of_trades",
        "taker_buy_base_volume", "taker_buy_quote_volume", "ignore"
    ])
    
    # Converting  timestamps to datetime
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
    
    # Converting  price and volume columns to numeric
    numeric_columns = ["open", "high", "low", "close", "volume", 
                      "quote_asset_volume", "taker_buy_base_volume", 
                      "taker_buy_quote_volume"]
    
    for col in numeric_columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Removal of  unnecessary columns
    df = df.drop(['ignore', 'close_time'], axis=1)
    
    print(f"✓ DataFrame created with shape: {df.shape}")
    return df

def handle_missing_values(df):
    """
    Handle missing values in the dataset
    """
    print("\nHandling missing values...")
    
    missing_before = df.isnull().sum().sum()
    print(f"Missing values before: {missing_before}")
    
    # Forward fill for time series data
    df = df.fillna(method='ffill')
    
    # Backward fill for any remaining NaN at the start
    df = df.fillna(method='bfill')
    
    missing_after = df.isnull().sum().sum()
    print(f"Missing values after: {missing_after}")
    
    return df

def remove_outliers(df, columns=['close', 'volume'], z_threshold=3):
    """
    Remove outliers using Z-score method
    """
    print(f"\nRemoving outliers (Z-score threshold: {z_threshold})...")
    
    rows_before = len(df)
    
    for col in columns:
        # Calculate Z-score
        z_scores = np.abs((df[col] - df[col].mean()) / df[col].std())
        
        # Keep only rows within threshold
        df = df[z_scores < z_threshold]
    
    rows_after = len(df)
    rows_removed = rows_before - rows_after
    
    print(f"✓ Removed {rows_removed} outlier rows ({(rows_removed/rows_before)*100:.2f}%)")
    print(f"Dataset size: {rows_before} → {rows_after}")
    
    return df.reset_index(drop=True)

def add_technical_features(df):
    """
    Add technical indicators and features for time-series forecasting
    """
    print("\nAdding technical features...")
    
    # Moving Averages
    df['MA_7'] = df['close'].rolling(window=7).mean()
    df['MA_14'] = df['close'].rolling(window=14).mean()
    df['MA_30'] = df['close'].rolling(window=30).mean()
    
    # Exponential Moving Averages
    df['EMA_12'] = df['close'].ewm(span=12).mean()
    df['EMA_26'] = df['close'].ewm(span=26).mean()
    
    # Price changes
    df['price_change'] = df['close'].diff()
    df['price_change_pct'] = df['close'].pct_change() * 100
    
    # Volatility (rolling standard deviation)
    df['volatility'] = df['close'].rolling(window=7).std()
    
    # High-Low spread
    df['hl_spread'] = df['high'] - df['low']
    df['hl_spread_pct'] = (df['hl_spread'] / df['close']) * 100
    
    # Volume features
    df['volume_ma_7'] = df['volume'].rolling(window=7).mean()
    df['volume_change_pct'] = df['volume'].pct_change() * 100
    
    # RSI (Relative Strength Index)
    delta = df['close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
    rs = gain / loss
    df['RSI'] = 100 - (100 / (1 + rs))
    
    # MACD
    df['MACD'] = df['EMA_12'] - df['EMA_26']
    df['MACD_signal'] = df['MACD'].ewm(span=9).mean()
    
    # Bollinger Bands
    df['BB_middle'] = df['close'].rolling(window=20).mean()
    bb_std = df['close'].rolling(window=20).std()
    df['BB_upper'] = df['BB_middle'] + (bb_std * 2)
    df['BB_lower'] = df['BB_middle'] - (bb_std * 2)
    
    # Time-based features
    df['hour'] = df['open_time'].dt.hour
    df['day_of_week'] = df['open_time'].dt.dayofweek
    df['day_of_month'] = df['open_time'].dt.day
    df['month'] = df['open_time'].dt.month
    
    print(f"✓ Added {len(df.columns) - 11} technical features")
    
    return df

def normalize_data(df, columns_to_normalize=None):
    """
    Normalize numerical features using MinMaxScaler
    """
    print("\nNormalizing data...")
    
    if columns_to_normalize is None:
        # Automatically select numeric columns (excluding time features)
        columns_to_normalize = df.select_dtypes(include=[np.number]).columns.tolist()
        columns_to_normalize = [col for col in columns_to_normalize 
                               if col not in ['hour', 'day_of_week', 'day_of_month', 'month']]
    
    scaler = MinMaxScaler()
    df_normalized = df.copy()
    
    df_normalized[columns_to_normalize] = scaler.fit_transform(df[columns_to_normalize])
    
    print(f"✓ Normalized {len(columns_to_normalize)} features")
    
    return df_normalized, scaler

def split_dataset(df, train_ratio=0.7, val_ratio=0.15):
    """
    Split dataset into training, validation, and testing sets
    """
    print("\nSplitting dataset...")
    
    # Remove rows with NaN values (from rolling calculations)
    df_clean = df.dropna().reset_index(drop=True)
    
    total_size = len(df_clean)
    train_size = int(total_size * train_ratio)
    val_size = int(total_size * val_ratio)
    
    # Split data chronologically (important for time series)
    train_data = df_clean[:train_size]
    val_data = df_clean[train_size:train_size + val_size]
    test_data = df_clean[train_size + val_size:]
    
    print(f"✓ Training set:   {len(train_data)} samples ({train_ratio*100:.0f}%)")
    print(f"✓ Validation set: {len(val_data)} samples ({val_ratio*100:.0f}%)")
    print(f"✓ Testing set:    {len(test_data)} samples ({(1-train_ratio-val_ratio)*100:.0f}%)")
    
    return train_data, val_data, test_data

def save_datasets(df_raw, df_processed, train_data, val_data, test_data):
    """
    Save all datasets to CSV files
    """
    print("\nSaving datasets...")
    
    # Save raw data
    df_raw.to_csv(RAW_DATA_FILE, index=False)
    print(f"✓ Raw data saved to: {RAW_DATA_FILE}")
    
    # Save processed data
    df_processed.to_csv(PROCESSED_DATA_FILE, index=False)
    print(f"✓ Processed data saved to: {PROCESSED_DATA_FILE}")
    
    # Save splits
    train_data.to_csv("crypto_train_data.csv", index=False)
    val_data.to_csv("crypto_val_data.csv", index=False)
    test_data.to_csv("crypto_test_data.csv", index=False)
    print(f"✓ Train/Val/Test splits saved")

def print_summary(df):
    """
    Print dataset summary statistics
    """
    print("\n" + "="*50)
    print("DATASET SUMMARY")
    print("="*50)
    print(f"\nShape: {df.shape}")
    print(f"Date range: {df['open_time'].min()} to {df['open_time'].max()}")
    print(f"\nPrice Statistics:")
    print(df['close'].describe())
    print(f"\nVolume Statistics:")
    print(df['volume'].describe())
    print(f"\nAll columns ({len(df.columns)}):")
    print(df.columns.tolist())

# ==================== MAIN EXECUTION ====================
def main():
    print("\n" + "="*50)
    print("CRYPTOCURRENCY DATA COLLECTION & PREPROCESSING")
    print("="*50)
    
    # Step 1: Download data
    klines = download_historical_data(SYMBOL, INTERVAL, START_DATE)
    
    # Step 2: Create DataFrame
    df_raw = create_dataframe(klines)
    
    # Step 3: Handle missing values
    df = handle_missing_values(df_raw)
    
    # Step 4: Remove outliers
    df = remove_outliers(df)
    
    # Step 5: Add technical features
    df = add_technical_features(df)
    
    # Step 6: Normalize data (optional - uncomment if needed)
    # df_normalized, scaler = normalize_data(df)
    
    # Step 7: Split dataset
    train_data, val_data, test_data = split_dataset(df)
    
    # Step 8: Save all datasets
    save_datasets(df_raw, df, train_data, val_data, test_data)
    
    # Step 9: Print summary
    print_summary(df)
    
    print("\n" + "="*50)
    print("✓ DATA PREPROCESSING COMPLETE!")
    print("="*50)
    print("\nFiles created:")
    print("1. crypto_raw_data.csv - Original downloaded data")
    print("2. crypto_processed_data.csv - Full processed dataset")
    print("3. crypto_train_data.csv - Training set (70%)")
    print("4. crypto_val_data.csv - Validation set (15%)")
    print("5. crypto_test_data.csv - Testing set (15%)")
    print("\nReady for model training in Week 3-4!")

if __name__ == "__main__":
    main()


CRYPTOCURRENCY DATA COLLECTION & PREPROCESSING

Downloaded 10000 records...
Downloaded 20000 records...
Downloaded 30000 records...
Downloaded 40000 records...
Downloaded 50000 records...

✓ Total records downloaded: 51654

Creating DataFrame...
✓ DataFrame created with shape: (51654, 10)

Handling missing values...
Missing values before: 0
Missing values after: 0

Removing outliers (Z-score threshold: 3)...
✓ Removed 1076 outlier rows (2.08%)
Dataset size: 51654 → 50578

Adding technical features...
✓ Added 21 technical features

Splitting dataset...
✓ Training set:   35384 samples (70%)
✓ Validation set: 7582 samples (15%)
✓ Testing set:    7583 samples (15%)

Saving datasets...
✓ Raw data saved to: crypto_raw_data.csv
✓ Processed data saved to: crypto_processed_data.csv
✓ Train/Val/Test splits saved

DATASET SUMMARY

Shape: (50578, 32)
Date range: 2020-01-01 00:00:00 to 2025-11-23 13:00:00

Price Statistics:
count     50578.000000
mean      46950.373202
std       31288.188408
min  